In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName("BigDataCommerceAnalytics")
    .master("local[*]")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.5.0")
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017")
    .getOrCreate())

print("Spark", spark.version)

:: loading settings :: url = jar:file:/home/psychoreveuse/COURS/BIGDATA/bigdata_tp/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/psychoreveuse/.ivy2.5.2/cache
The jars for the packages stored in: /home/psychoreveuse/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c8b8327e-02c6-4c25-83dc-a39f3de06268;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;10.5.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
downloading https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.13/10.5.0/mongo-spark-connector_2.13-10.5.0.jar ...
	[SUCCESSFUL ]

Spark 4.0.1


In [2]:
df_commandes = (spark.read.format("mongodb")
    .option("database", "commerce")
    .option("collection", "raw_commandes")
    .load())

df_paiements = (spark.read.format("mongodb")
    .option("database", "commerce")
    .option("collection", "raw_payments")
    .load())

df_commandes.printSchema()
df_commandes.show(5)
print("Commandes :", df_commandes.count())

df_paiements.printSchema()
df_paiements.show(5)
print("Paiements :", df_paiements.count())

root
 |-- _id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cart_discounted_total: double (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- cart_total: double (nullable = true)
 |-- category: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- discounted_line_total: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)



+--------------------+--------------+---------------------+-------+----------+------------------+-------------------+---------------------+-----------------+----------+----------+----------+--------------------+--------+----------+-------+--------+
|                 _id|         brand|cart_discounted_total|cart_id|cart_total|          category|discount_percentage|discounted_line_total|       line_total|order_date|  order_id|product_id|       product_title|quantity|unit_price|user_id|username|
+--------------------+--------------+---------------------+-------+----------+------------------+-------------------+---------------------+-----------------+----------+----------+----------+--------------------+--------+----------+-------+--------+
|6b0808ba5ec600a7c...|       inconnu|             11510.81|      1|  13037.88|              tops|              12.13|               105.41|           119.96|2025-03-02|CART-00001|       162|          Blue Frock|       4|     29.99|      1|  emilys|
|6b0

In [3]:
resultat = (df_commandes
    .select("category", "quantity", "discounted_line_total")
    .filter(F.col("quantity") >= 2)
    .withColumn("montant_arrondi", F.round(F.col("discounted_line_total"), 2))
    .groupBy("category")
    .agg(F.sum("montant_arrondi").alias("ca"), F.count("*").alias("nb_lignes"))
    .orderBy(F.desc("ca")))

resultat.show()
print("Nombre de catégories :", resultat.count())

+-------------------+------------------+---------+
|           category|                ca|nb_lignes|
+-------------------+------------------+---------+
|            vehicle|1595455.2800000003|       16|
|         motorcycle| 594708.4299999999|       21|
|       mens-watches| 544863.8999999999|       20|
|     womens-watches|194369.12999999998|       10|
|            laptops|           82138.1|       14|
|          furniture| 74802.31999999999|       19|
|        smartphones|          68637.02|       47|
| mobile-accessories|24626.460000000003|       57|
|            tablets|14083.739999999998|       12|
|         mens-shoes| 8021.060000000001|       21|
|kitchen-accessories|           5207.21|       96|
|        womens-bags|           5175.59|       18|
|         fragrances|            4464.0|       17|
|     womens-dresses|3692.7699999999995|       14|
| sports-accessories| 3497.950000000001|       45|
|    home-decoration|           2556.55|       22|
|               tops|          

In [4]:
etape = (df_commandes
    .filter(F.col("category") == "smartphones")
    .withColumn("quantite_x2", F.col("quantity") * 2)
    .select("product_title", "quantity", "quantite_x2"))

print("Transformations définies")
etape.explain()

Transformations définies
== Physical Plan ==
*(1) Project [product_title#12, quantity#13, (quantity#13 * 2) AS quantite_x2#362]
+- BatchScan MongoTable()[product_title#12, quantity#13] MongoScan{namespaceDescription=commerce.raw_commandes} RuntimeFilters: []




In [5]:
etape.show()

+------------------+--------+-----------+
|     product_title|quantity|quantite_x2|
+------------------+--------+-----------+
|          iPhone 6|       3|          6|
|     iPhone 13 Pro|       1|          2|
|          Realme X|       1|          2|
|         Realme XT|       1|          2|
|          Vivo X21|       1|          2|
| Samsung Galaxy S8|       3|          6|
|     iPhone 13 Pro|       5|         10|
|     iPhone 13 Pro|       4|          8|
|           Oppo K1|       5|         10|
|           Vivo S1|       4|          8|
|        Realme C35|       2|          4|
|          iPhone 6|       3|          6|
|           Vivo V9|       2|          4|
| Oppo F19 Pro Plus|       1|          2|
|           Oppo K1|       4|          8|
|          Vivo X21|       2|          4|
| Samsung Galaxy S8|       4|          8|
|         iPhone 5s|       3|          6|
|Samsung Galaxy S10|       5|         10|
|Samsung Galaxy S10|       4|          8|
+------------------+--------+-----

In [6]:
print("Nombre de partitions :", df_commandes.rdd.getNumPartitions())

repartition = df_commandes.repartition(4)
print("Après repartition :", repartition.rdd.getNumPartitions())

(repartition
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .orderBy("partition")
    .show())

Nombre de partitions : 1
Après repartition : 4
+---------+-----+
|partition|count|
+---------+-----+
|        0|  200|
|        1|  200|
|        2|  200|
|        3|  200|
+---------+-----+



### Observation de Spark UI

Le `show()` de l'étape « partitions » a déclenché plusieurs jobs successifs.
Le job 14 contient un stage de 4 tasks (stage 21) : une task par partition de 200 lignes,
après le `repartition(4)`. Il écrit 248 B de résultats intermédiaires (shuffle write).
Le job 15 relit ces 248 B (shuffle read) dans un stage final d'une seule task (stage 24),
qui rassemble les 4 résultats partiels.

Les stages marqués « Skipped » ont déjà été calculés par un job précédent :
Spark réutilise leurs résultats de shuffle au lieu de les recalculer.

**Lien entre les notions :** une action déclenche un ou plusieurs jobs ; chaque job est
découpé en stages, séparés par les shuffles (`repartition`, `groupBy`) ; chaque stage
exécute une task par partition.

### Atelier 3

In [7]:
ca_categorie = (df_commandes
    .groupBy("category")
    .agg(
        F.round(F.sum("discounted_line_total"), 2).alias("ca"),
        F.count("*").alias("nb_lignes")
    )
    .orderBy(F.desc("ca")))

ca_categorie.show(24)

+-------------------+----------+---------+
|           category|        ca|nb_lignes|
+-------------------+----------+---------+
|            vehicle|1682064.05|       19|
|         motorcycle| 609460.61|       23|
|       mens-watches| 593063.86|       26|
|     womens-watches| 249189.51|       18|
|            laptops|  85006.24|       16|
|          furniture|  77674.97|       22|
|        smartphones|  73130.95|       60|
| mobile-accessories|  26160.99|       70|
|            tablets|  14890.16|       14|
|         mens-shoes|    8278.9|       23|
|        womens-bags|   5849.91|       21|
|kitchen-accessories|   5714.33|      119|
|         fragrances|   4958.05|       24|
|     womens-dresses|   4064.13|       19|
| sports-accessories|   3776.68|       60|
|    home-decoration|    2747.3|       28|
|               tops|   2233.24|       25|
|          groceries|    1881.5|      113|
|       womens-shoes|   1538.66|       15|
|         sunglasses|   1509.44|       17|
|        me

**Commentaire – CA par catégorie**

Le chiffre d'affaires est calculé à partir de `discounted_line_total` (montant après remise),
sans utiliser `cart_total` qui est répété sur chaque ligne d'un panier. La somme des 24
catégories (3 456 709,58) est égale au montant total des 208 paiements simulés, ce qui
confirme l'absence de double comptage.

Le CA est très concentré : `vehicle` représente environ 49 % du total, et les trois
premières catégories (vehicle, motorcycle, mens-watches) environ 83 %. Les catégories les
plus fréquentes (kitchen-accessories, groceries) pèsent peu : le nombre de lignes ne
reflète pas la valeur. Cet indicateur porte sur tous les paniers, quel que soit le statut
du paiement : il ne s'agit pas du CA encaissé.

In [8]:
top_clients = (df_paiements
    .filter(F.col("status") == "PAID")
    .groupBy("user_id")
    .agg(F.round(F.sum("amount"), 2).alias("montant_paye"))
    .orderBy(F.desc("montant_paye"))
    .limit(10))

top_clients.show()

+-------+------------+
|user_id|montant_paye|
+-------+------------+
|     30|   155507.16|
|     52|   130175.28|
|     83|    125679.0|
|     33|   125461.58|
|     45|   121385.95|
|    155|   110543.89|
|     78|   109890.08|
|    198|    92308.32|
|    194|    83615.86|
|    123|    78514.37|
+-------+------------+



**Commentaire – Top 10 clients**

Le classement ne retient que les paiements PAID, seuls considérés comme encaissés.
Les 10 premiers clients totalisent 1 133 081,49, soit environ 49 % du montant encaissé
(2 322 729,73), alors qu'ils ne représentent que 10 des 145 clients ayant payé
(moins de 7 %). Le chiffre d'affaires dépend donc fortement d'un petit nombre de clients.

Limite : dans DummyJSON, chaque utilisateur n'a qu'un seul panier. Ce classement est donc
aussi celui des 10 plus gros paniers payés, et non un vrai historique de fidélité client.

In [9]:
total = df_paiements.count()
refus = df_paiements.filter(F.col("status") == "REFUSED").count()
print(f"Taux de refus : {refus} / {total} = {refus / total:.1%}")

(df_paiements
    .groupBy("status")
    .count()
    .withColumn("pourcentage", F.round(F.col("count") / total * 100, 1))
    .orderBy(F.desc("count"))
    .show())

Taux de refus : 42 / 208 = 20.2%
+-------+-----+-----------+
| status|count|pourcentage|
+-------+-----+-----------+
|   PAID|  145|       69.7|
|REFUSED|   42|       20.2|
|PENDING|   21|       10.1|
+-------+-----+-----------+



**Commentaire – Taux de refus**

Sur 208 tentatives de paiement, 42 ont été refusées, soit un taux de refus de 20,2 %
(cible de la simulation : 20 %). La répartition complète (69,7 % PAID, 20,2 % REFUSED,
10,1 % PENDING) est très proche des probabilités imposées : les écarts viennent du
tirage aléatoire sur un petit nombre de paniers. En montant, les paiements refusés et en
attente représentent environ 1 134 000 non encaissés, soit un tiers du montant total des
paniers.

Limite : ce taux est simulé et ne reflète pas un comportement réel. L'indicateur « par
pays » prévu dans la diapo n'est pas calculable : les commandes ne contiennent pas de pays.

In [10]:
dates_paniers = (df_commandes
    .groupBy("cart_id")
    .agg(F.min(F.to_date("order_date")).alias("date_panier")))

mensuel = (df_paiements
    .filter(F.col("status") == "PAID")
    .join(dates_paniers, on="cart_id")
    .withColumn("mois", F.date_format("date_panier", "yyyy-MM"))
    .groupBy("mois")
    .agg(
        F.round(F.sum("amount"), 2).alias("encaisse"),
        F.count("*").alias("nb_paiements")
    )
    .orderBy("mois"))

mensuel.show(12)

+-------+----------+------------+
|   mois|  encaisse|nb_paiements|
+-------+----------+------------+
|2025-01|1053546.98|          38|
|2025-02| 183804.76|          23|
|2025-03| 286359.73|          24|
|2025-04|  79188.71|          17|
|2025-05|  178750.2|          15|
|2025-06| 278410.54|          11|
|2025-07|  15897.94|           4|
|2025-08| 142184.91|           6|
|2025-09|  63319.54|           4|
|2025-10|  41266.42|           3|
+-------+----------+------------+



**Commentaire – Évolution mensuelle**

Les paiements n'ont pas de date : chaque paiement PAID a été rattaché à son panier
(jointure sur `cart_id`), en retenant la première date des lignes du panier (`min` de
`order_date`, convertie en date). Le total des 10 mois (2 322 729,73 pour 145 paiements)
correspond bien au montant encaissé.

La forte baisse observée (plus d'un million en janvier, quasiment rien en fin d'année,
aucun paiement en novembre et décembre) n'est pas une tendance réelle. Elle vient de la
méthode : les dates des lignes étant tirées au hasard, la plus petite date d'un panier de
plusieurs lignes tombe presque toujours en début d'année. Les dates étant de plus
synthétiques, cet indicateur ne permet aucune conclusion sur la saisonnalité. Il montre
surtout qu'un choix de traitement peut créer une tendance artificielle.